# 02 — Behavioral EDA and Hypothesis Generation

## Purpose

This notebook documents the exploratory stage of the research process.

In the original real-world workflow, interactive Power BI analysis was used to rapidly examine repayment behavior across different dimensions. The purpose was to discover patterns that deserved deeper statistical investigation.

The public notebook recreates that reasoning against the synthetic dataset generated in **Notebook 01**.

### Research question

> **Which early repayment behaviors appear to distinguish loans that eventually default from loans that eventually settle successfully?**

### Important distinction

This notebook is for **exploration and hypothesis generation**.

Formal statistical tests come later. An observed pattern here is therefore treated as a **candidate signal**, not yet as proof of statistical significance or causality.

## 1. Load the synthetic early-life modeling dataset

Notebook 01 generates:

`data/synthetic/synthetic_early_modeling_base.csv`

The dataset contains the variables intended to be available at the early prediction point together with the eventual outcome.

The full-lifecycle variables used internally to generate the synthetic target are not used as predictors here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/synthetic/synthetic_early_modeling_base.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

## 2. Initial data audit

Before exploring relationships, inspect:

- missingness;
- data types;
- target availability;
- repayment-frequency coverage;
- basic ranges of behavioral variables.

This reflects the broader principle that **data quality and representation decisions come before model fitting**.

In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True)
}).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

audit

In [ ]:
target_summary = (
    df["is_good_or_bad"]
    .value_counts(dropna=False)
    .rename(index={0: "Good (0)", 1: "Default (1)"})
    .to_frame("count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(df) * 100
).round(2)

target_summary

## 3. Outcome distribution across repayment frequency

Repayment frequency is structurally important because the same number of calendar days has different meaning under weekly, bi-weekly, and monthly schedules.

This motivates later investigation of **cycle-normalized overdue measures** rather than relying only on raw calendar duration.

In [ ]:
frequency_default = (
    df.groupby("frequency_name", observed=True)["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
)

frequency_default["default_rate_pct"] = (
    frequency_default["default_rate"] * 100
).round(2)

frequency_default.sort_values("default_rate_pct", ascending=False)

In [ ]:
ax = (
    frequency_default["default_rate_pct"]
    .sort_values(ascending=False)
    .plot(kind="bar", figsize=(8, 4))
)

ax.set_title("Exploratory Default Rate by Repayment Frequency")
ax.set_xlabel("Repayment Frequency")
ax.set_ylabel("Eventual Default Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. First behavioral signal: missed installments

A natural starting point is the number of missed installments observed during the early period.

Rather than immediately fitting a classifier, inspect whether the **eventual default rate changes as early missed-installment behavior increases**.

This corresponds to the operational question:

> Does early repayment deterioration contain information about the eventual outcome?

In [ ]:
missed_rate = (
    df.groupby("early_missed_installment_count")["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

missed_rate["default_rate_pct"] = (
    missed_rate["default_rate"] * 100
).round(2)

missed_rate

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(
    missed_rate["early_missed_installment_count"],
    missed_rate["default_rate_pct"],
    marker="o"
)
plt.title("Exploratory Default Rate by Early Missed-Installment Count")
plt.xlabel("Early missed-installment count")
plt.ylabel("Eventual default rate (%)")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Exploratory interpretation

The objective is to look for a directional relationship.

A consistent increase in default rate with missed-installment frequency would justify carrying the variable into formal statistical analysis and model development.

This remains an **exploratory observation**, not a causal conclusion.

## 5. Consecutive versus isolated misses

The original exploratory work went beyond the total number of missed installments. A key question was whether missed payments occurred **consecutively**.

Two loans can have the same number of missed installments but very different behavioral trajectories:

- isolated misses followed by recovery;
- repeated consecutive misses indicating persistent repayment stress.

For the synthetic data, a maximum consecutive-miss value of at least two is treated as evidence of a consecutive-miss episode.

In [ ]:
df["has_consecutive_miss"] = (
    df["early_max_consecutive_missed"] >= 2
).astype(int)

consecutive_rate = (
    df.groupby("has_consecutive_miss")["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
)

consecutive_rate["default_rate_pct"] = (
    consecutive_rate["default_rate"] * 100
).round(2)

consecutive_rate.index = consecutive_rate.index.map({
    0: "No consecutive miss",
    1: "At least one consecutive-miss episode"
})

consecutive_rate

### Research hypothesis H1

> **Persistent early delinquency:** Loans exhibiting consecutive missed installments during the early observation period may have a higher probability of eventual default than loans with only isolated misses.

This hypothesis will be investigated formally in the statistical-analysis stage.

## 6. Recovery behavior after a missed payment

Another behavioral question is **how quickly missed payments are recovered**.

The synthetic process records recovery delay in installment cycles. This provides a reproducible proxy for the broader real-world question explored during interactive analysis: whether missed payments were commonly recovered by the next scheduled cycle or recovered later.

The purpose is to investigate whether the *timing* of recovery carries information beyond the simple count of missed installments.

In [ ]:
recovery_rate = (
    df.groupby("early_recovery_delay_cycles")["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

recovery_rate["default_rate_pct"] = (
    recovery_rate["default_rate"] * 100
).round(2)

recovery_rate

In [ ]:
plt.figure(figsize=(8.5, 4.5))
plt.plot(
    recovery_rate["early_recovery_delay_cycles"],
    recovery_rate["default_rate_pct"],
    marker="o"
)
plt.title("Exploratory Default Rate by Early Recovery Delay")
plt.xlabel("Cumulative recovery-delay cycles")
plt.ylabel("Eventual default rate (%)")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Research hypothesis H2

> **Recovery persistence:** Slower recovery from missed payments may be associated with higher eventual-default risk.

The important methodological point is that repayment behavior is being represented as a **trajectory**, rather than collapsing everything into a single missed-payment count.

## 7. Why normalize overdue duration?

A central representation problem is that raw overdue days are not directly comparable across heterogeneous repayment schedules.

For example, an identical calendar-day delay corresponds to different numbers of repayment cycles under weekly, bi-weekly, and monthly schedules.

The project therefore investigates:

**overdue cycles = overdue days / installment-frequency interval**

This is a representation motivated by comparability across repayment schedules.

In [ ]:
df["pass_due_cycle_ratio"] = (
    df["early_max_overdue_days"]
    / df["installment_days"]
)

df[[
    "frequency_name",
    "early_max_overdue_days",
    "installment_days",
    "pass_due_cycle_ratio"
]].head(10)

## 8. Default rate across normalized overdue cycles

The next question is whether the normalized measure provides a clearer behavioral relationship with eventual default than raw overdue days.

Broad bins are used here so the plot represents **behavioral stages** rather than every individual numeric value.

In [ ]:
bins = [-0.001, 0, 0.5, 1, 1.5, 2, 3, np.inf]
labels = [
    "0",
    "0–0.5",
    "0.5–1",
    "1–1.5",
    "1.5–2",
    "2–3",
    ">3"
]

df["pass_due_cycle_bin"] = pd.cut(
    df["pass_due_cycle_ratio"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
)

cycle_rate = (
    df.groupby("pass_due_cycle_bin", observed=False)["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

cycle_rate["default_rate_pct"] = (
    cycle_rate["default_rate"] * 100
).round(2)

cycle_rate

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(
    cycle_rate["pass_due_cycle_bin"].astype(str),
    cycle_rate["default_rate_pct"],
    marker="o"
)
plt.title("Exploratory Default Rate by Normalized Overdue Cycles")
plt.xlabel("Overdue duration / installment interval")
plt.ylabel("Eventual default rate (%)")
plt.xticks(rotation=30)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Frequency-stratified view

A normalized feature is most useful if its interpretation is not dominated by one repayment frequency.

The following table gives the exploratory default rate by normalized overdue-cycle band within each repayment schedule.

In [ ]:
frequency_cycle = (
    df.groupby(
        ["frequency_name", "pass_due_cycle_bin"],
        observed=False
    )["is_good_or_bad"]
    .mean()
    .mul(100)
    .round(2)
    .rename("default_rate_pct")
    .reset_index()
)

frequency_cycle

### Research hypothesis H3

> **Temporal normalization:** Cycle-normalized overdue persistence may provide a more comparable behavioral signal across heterogeneous repayment schedules than raw overdue days.

This representation will be investigated further before it is accepted as a modeling feature.

## 9. Relative overdue burden

The exploratory workflow also examined overdue amount relative to scheduled repayment.

The synthetic version uses:

> **overdue proportion = early overdue amount proxy / scheduled repayment amount**

The purpose is to ask whether **relative repayment burden** contains a clearer signal than raw currency amounts.

In [ ]:
plt.figure(figsize=(9, 5))

binned_overdue = pd.qcut(
    df["overdue_proportion"],
    q=10,
    duplicates="drop"
)

overdue_rate = (
    df.groupby(binned_overdue, observed=True)["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

overdue_rate["default_rate_pct"] = (
    overdue_rate["default_rate"] * 100
).round(2)

plt.plot(
    range(len(overdue_rate)),
    overdue_rate["default_rate_pct"],
    marker="o"
)
plt.title("Exploratory Default Rate Across Overdue-Proportion Deciles")
plt.xlabel("Overdue-proportion decile")
plt.ylabel("Eventual default rate (%)")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

overdue_rate

### Research hypothesis H4

> **Relative delinquency burden:** A higher proportion of scheduled repayment represented by early overdue amounts may be associated with higher eventual-default risk.

The motivation is comparability across loans with different installment amounts.

## 10. Missed-installment proportion

Raw missed-installment counts depend partly on how many installments occur before the prediction point.

A normalized alternative is:

> **missed-installment proportion = early missed installments / installments observed before the prediction point**

This asks whether the *rate* of missed repayment cycles is more informative than the absolute count.

In [ ]:
missed_prop_bins = pd.qcut(
    df["missed_installment_proportion"],
    q=8,
    duplicates="drop"
)

missed_prop_summary = (
    df.groupby(missed_prop_bins, observed=True)["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

missed_prop_summary["default_rate_pct"] = (
    missed_prop_summary["default_rate"] * 100
).round(2)

missed_prop_summary

## 11. From visual patterns to research questions

The exploratory stage should leave the project with a small number of **testable hypotheses**, not an unbounded list of variables.

| Hypothesis | Behavioral representation | Next stage |
|---|---|---|
| H1 | Consecutive missed installments | Statistical association + modeling |
| H2 | Recovery delay after missed payments | Statistical association + modeling |
| H3 | Normalized overdue cycles | Representation comparison + modeling |
| H4 | Relative overdue burden | Statistical association + modeling |
| H5 | Missed-installment proportion | Compare normalized vs raw count |

These questions are deliberately connected to the project's research objective rather than chosen only because a variable is available.

## 12. Research decisions from EDA

### Carry forward for deeper investigation

- early missed-installment count;
- consecutive-miss behavior;
- recovery timing;
- normalized overdue duration;
- relative overdue burden;
- missed-installment proportion;
- repayment frequency as a heterogeneity dimension.

### Not concluded at this stage

EDA does not establish statistical significance, robustness to confounding, or out-of-sample predictive value.

Those questions belong to the subsequent statistical and modeling stages.

> **EDA generates hypotheses; it does not validate them.**

## 13. Connection to the broader analytical workflow

The exploratory stage follows the same reasoning pattern used in the original project:

```text
Interactive / descriptive exploration
                ↓
        Pattern discovery
                ↓
       Research hypothesis
                ↓
      Statistical analysis
                ↓
       Feature engineering
                ↓
       Predictive modeling
```

Power BI can be used for rapid interactive exploration in the real-world workflow; the public notebook provides the reproducible Python equivalent over synthetic data.

The next notebook formalizes the candidate relationships using statistical analysis.